In [49]:
!pip install datasets transformers pandas
!pip install nltk

PIPELINE de TRAITEMET des textes

In [50]:
import pandas as pd
from google.colab import drive
import re

# Monter Google Drive
drive.mount('/content/drive')

# Charger le CSV avec les mots et leurs racines
df = pd.read_csv("/content/drive/MyDrive/Analyzer-sentiment/malagasy_dictionary.csv")
print(df.head())

# Créer le dictionnaire dérivé -> racine
derive_to_racine = dict(zip(df['mot'], df['racine']))
print(derive_to_racine)

# Stopwords malagasy
stopwords_malagasy = [
    "ny", "dia", "ao", "amin'", "ka", "sy", "izay", "raha", "ho", "tsy", "izy", "azy",
    "izahay", "ianao", "ianao rehetra", "isika", "ity", "izany", "ireo", "rehetra", "teo",
    "tamin'", "tamin'ny", "indray", "ihany", "an'", "amin'ny", "eo", "ary", "aza",
    "rehefa", "moa", "koa", "na", "raha tsy", "amin'", "ho an'ny", "amin'ny fotoana",
    "satria", "eto", "any"
]

import unicodedata

def preprocess_text(text):
    # Minuscules
    text = text.lower()

    # Normalisation Unicode
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

    # Supprimer ponctuation et chiffres
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", "", text)

    # Réduction des répétitions de lettres
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Tokenisation plus robuste
    tokens = re.findall(r"\b\w+\b", text)

    # Filtrage stopwords et mots courts
    tokens = [t for t in tokens if t not in stopwords_malagasy and len(t) > 2]

    # Stemming / racinisation via dictionnaire
    for i, t in enumerate(tokens):
        t = derive_to_racine.get(t, t)
        # fallback simple si non trouvé
        for suffix in ["ina", "ana", "ena", "any"]:
            if t.endswith(suffix) and len(t) > len(suffix)+2:
                t = t[:-len(suffix)]
        tokens[i] = t

    return " ".join(tokens)

# Test
print(preprocess_text("Ny fiainana dia tsara sy mahafinaritra! miaina hatsara"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  racine         mot
0    aby      abiaby
1  abily    abiliana
2  abily     fiabily
3  abily  fiabiliana
4  abily   habiliana
{'abiaby': 'aby', 'abiliana': 'abily', 'fiabily': 'abily', 'fiabiliana': 'abily', 'habiliana': 'abily', 'iabiliana': 'abily', 'miabily': 'abily', 'mpiabily': 'abily', 'aboabo': 'abo', 'aboina': 'abo', 'aboÃ±abo': 'abo', 'fanabo': 'abo', 'haabo': 'abo', 'haboa': 'abo', 'haboana': 'abo', 'habosana': 'abo', 'mahaabo': 'abo', 'manabo': 'abo', 'miabo': 'abo', 'mihaabo': 'abo', 'piabo': 'abo', 'aboaboina': 'aboabo', 'anaboaboana': 'aboabo', 'fanaboabo': 'aboabo', 'fanaboaboana': 'aboabo', 'fiaboabo': 'aboabo', 'fiaboaboana': 'aboabo', 'iaboaboana': 'aboabo', 'manaboabo': 'aboabo', 'miaboabo': 'aboabo', 'mihaboabo': 'aboabo', 'mpanaboabo': 'aboabo', 'mpiaboabo': 'aboabo', 'piaboabo': 'aboabo', 'voaboabo': 'aboabo', 'aboboina': 'abobo', 'maÃ±a

Chargement du dataset et prétraitement du dataset pour l'aalyse de sentiment

In [51]:
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder

# Charger le dataset
dataset = load_dataset("Lo-Renz-O/vaovao_malagasy_sentiment_corpus")

# Appliquer le preprocessing sur toutes les phrases
def clean_example(example):
    example['clean_text'] = preprocess_text(example['text'])
    return example

dataset = dataset.map(clean_example)

# Vérification
print(dataset['train'][0])


# Extraire les labels
y_train_text = [ex['label'] for ex in dataset['train']]
y_val_text = [ex['label'] for ex in dataset['validation']]

# Encoder : negative -> 0, positive -> 1
le = LabelEncoder()
y_train = le.fit_transform(y_train_text)
y_val = le.transform(y_val_text)



{'text': 'Dimy ao amin’ny faritra Diana.', 'label': 'negative', 'clean_text': 'dimy aminny faritra diana'}


Vectorsation des mots avec TF-IDF

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Extraire les textes prétraités
train_texts = [ex['clean_text'] for ex in dataset['train']]
val_texts = [ex['clean_text'] for ex in dataset['validation']]

# Fit TF-IDF sur le train, transformer train et validation
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

ENtrainement

In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=3000,
    C=2.0,
    solver='liblinear'
)
model.fit(X_train, y_train)

# Prédiction sur validation
y_pred = model.predict(X_val)

# Évaluation
print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred, target_names=le.classes_))

Accuracy: 0.7564356435643564
              precision    recall  f1-score   support

    negative       0.75      0.72      0.74       237
    positive       0.76      0.79      0.77       268

    accuracy                           0.76       505
   macro avg       0.76      0.75      0.75       505
weighted avg       0.76      0.76      0.76       505



Test

In [54]:
def predict_sentiment(text, model, vectorizer, le, preprocess_text):
    # Preprocessing
    clean_text = preprocess_text(text)

    # Transformation en vecteur
    X_new = vectorizer.transform([clean_text])

    # Prédiction
    prediction = model.predict(X_new)

    # Décoder le label
    return le.inverse_transform(prediction)[0]


text_test = "malahelo aho"
sentiment = predict_sentiment(text_test, model, vectorizer, le, preprocess_text)
print("Phrase test:", text_test)
print("Sentiment prédit:", sentiment)

Phrase test: malahelo aho
Sentiment prédit: negative


In [55]:
import pickle
with open("sentiment_model.pkl", "wb") as f:
    pickle.dump({
        'model': model,
        'vectorizer': vectorizer,
        'label_encoder': le
    }, f)


In [57]:

with open("sentiment_model.pkl", "rb") as f:
  data = pickle.load(f)

model = data['model']
vectorizer = data['vectorizer']
le = data['label_encoder']
text_test = " Tia anao Jehovah"
sentiment = predict_sentiment(text_test, model, vectorizer, le, preprocess_text)
print("Sentiment prédit:", sentiment)

Sentiment prédit: positive
